## Explore Dataset

In [55]:
import geopandas as gpd
import dotenv
import os
import pandas as pd
import shapely
from google.cloud import storage
from addCSOs import calculate_risk_factors_cso

In [2]:
dotenv.load_dotenv()
RIVERS_URL = os.getenv("RIVERS_URL")

## Obtaining data

Keeping all the data in the EPSG 3857 coordinate referencing system (CRS) to work better with distances

### CSO Data

In [3]:
geo_JSON_url = "https://services-eu1.arcgis.com/MSNNjkZ51iVh8yBj/arcgis/rest/services/Northumbrian_Water_Storm_Overflow_Activity_2_view/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
cso: gpd.GeoDataFrame = gpd.read_file(geo_JSON_url)
cso = cso.to_crs(epsg=3857)

In [56]:
INPUT_PATH = os.getenv("CSO_DATA_LOC")
if INPUT_PATH is None:
    raise ValueError("Input Path not set")
bucketName = INPUT_PATH.split("/")[2]
path_in_bucket = '/'.join(INPUT_PATH.split("/")[3:])
storage_client = storage.Client()
bucket = storage_client.get_bucket(bucketName)
geojson_filenames = ["gs://"+bucketName+'/' +
                     fname.name for fname in bucket.list_blobs(prefix=path_in_bucket) if fname.name.endswith('.geojson')]
cso = gpd.read_file(geojson_filenames[-1])
cso = cso.to_crs(epsg=3857)

### River data

In [4]:
rivers = gpd.read_file(RIVERS_URL)
rivers = rivers.to_crs(epsg=3857)

### Clip to cso bounds

In [7]:
rivers = rivers.clip(cso.total_bounds).explode()

### Catchment Data

In [5]:
cmnt = gpd.read_file(
    "https://environment.data.gov.uk/catchment-planning/OperationalCatchment/3052.geojson")
cmnt.to_crs(cso.crs,inplace=True)

In [82]:
rivers = gpd.clip(rivers, cmnt)
# Remove all rivers that aren't LineStrings after Clipping
rivers = rivers[rivers.geometry.apply(
    lambda x: isinstance(x, shapely.LineString))]
cso = gpd.clip(cso, cmnt)

In [83]:
m = cmnt.explore(highlight=False,
                 popup=False,
                 tooltip=False)
rivers.explore(m=m)
cso.explore(m=m, marker_kwds={"radius":5},color="red")

## Use addCSOs.py

In [74]:
RIVER_RESOLUTION:float= 50
MAX_OUTFALL_DIST_FROM_RIVER:float = 500
MAX_RISK_DISTANCE: float = 3000

In [109]:
river_points, cso_with_locations = calculate_risk_factors_cso(
    cso,
    rivers,
    cso_df_id_col="id",
    river_resolution=RIVER_RESOLUTION,
    max_outfall_dist_from_river=MAX_OUTFALL_DIST_FROM_RIVER,
    max_risk_distance=MAX_RISK_DISTANCE,
    include_upstream=False,
    river_directed=True,
)

## Add Risk

In [110]:
def simple_risk(csoInfo: list[tuple])->float:
    tot = 0
    for csoID, dist in csoInfo:
        tot += MAX_RISK_DISTANCE - abs(dist)
    return tot

def add_simple_risk(gdf:gpd.GeoDataFrame)->gpd.GeoSeries:
    return gdf["csoInfo"].apply(simple_risk)

In [122]:
def linear_dist_decrease(d:float)->float:
    return (MAX_RISK_DISTANCE-d)/MAX_RISK_DISTANCE

In [123]:
def add_frequency_risk(gdf:gpd.GeoDataFrame)->gpd.GeoSeries:
    EDM_PATH = os.getenv("CSO_DATA_OUTPUT")
    edm = pd.read_pickle(EDM_PATH)
    freq = edm.groupby("Monitor_ID").count()["Event_Start"]
    return gdf["csoInfo"].apply(lambda x: sum([freq[csoid]*linear_dist_decrease(dist) for csoid, dist in x if csoid in freq.index]))

In [124]:
def add_duration_risk(gdf:gpd.GeoDataFrame)->gpd.GeoSeries:
    EDM_PATH = os.getenv("CSO_DATA_OUTPUT")
    edm = pd.read_pickle(EDM_PATH)
    dur = edm["Event_End"] - edm["Event_Start"]
    duration = dur.groupby("Monitor_ID").sum()
    return gdf["csoInfo"].apply(lambda x: sum([duration[csoid].total_seconds()/60/60*linear_dist_decrease(dist) for csoid,dist in x if csoid in duration.index]))

In [125]:
river_points["Risk"] = add_frequency_risk(river_points)

In [126]:
river_points_risk = river_points[river_points["Risk"] > 0]

## Explore

In [127]:
m = cmnt.explore(
             highlight = False,
             tooltip = False,
             popup = False,
             )
rivers.explore(
    m=m,
    color="blue",
)
river_points_risk.explore(
    m=m,
    column="Risk",
    cmap="viridis",
    marker_kwds={"radius":5},
)
cso_with_locations.set_geometry("outfallLocation").explore(
    m=m,
    color="pink",
    marker_kwds={"radius": 8}
)